# **Cryptocurrency Volatility Prediction**
### PW Skills | Machine Learning Project

**Goal**: predict how volatile each cryptocurrency will be over the **next 7 days** using OHLC prices, volume and market cap.

**Steps in this notebook**: Data collection → Preprocessing → EDA → Feature engineering → Model selection and training → Evaluation → Hyper-parameter tuning → Testing on unseen data → Deployment.

> **How to run**: put the dataset CSV file(s) (Cryptocurrency Historical Prices) inside a folder called `data/` (in Colab: upload to `/content/data/`), then **Runtime → Run all**.
> All the pipeline code is in the cell below (it is saved to `src/crypto_volatility.py`), so this notebook is fully self-contained.

## **Step 0: Setup**

In [ ]:
import os
os.makedirs('src', exist_ok=True)
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')          # run from the repository root when used inside the repo
print('Working directory:', os.getcwd())

In [ ]:
%%writefile src/crypto_volatility.py
"""
crypto_volatility.py
====================
End-to-end pipeline for Cryptocurrency Volatility Prediction.

Stages (see docs/PIPELINE.md):
    load_data -> clean_data -> build_features -> time_split -> train_pipeline
    -> save bundle -> predict_latest (used by Streamlit app / Flask API)

Target definition
-----------------
    target_vol(t) = standard deviation of the daily log-returns over the NEXT
                    HORIZON days (t+1 ... t+HORIZON).
All features at day t only use information available up to day t, so there is
no look-ahead leakage.
"""
from __future__ import annotations

import glob
import json
import os
import re

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

HORIZON = 7            # predict volatility of the next 7 days
RANDOM_STATE = 42

# --------------------------------------------------------------------------
# 1. DATA LOADING
# --------------------------------------------------------------------------
BASE_COLS = ["date", "symbol", "open", "high", "low", "close", "volume", "market_cap"]

# normalised header (lower-case letters only) -> standard column name
_ALIASES = {
    "date": "date", "timestamp": "date", "datetime": "date",
    "symbol": "symbol", "ticker": "symbol", "name": "name", "coin": "name",
    "open": "open", "high": "high", "low": "low", "close": "close",
    "volume": "volume", "marketcap": "market_cap", "marketcapitalization": "market_cap",
    "mcap": "market_cap",
}


def _standardize(df: pd.DataFrame, default_symbol: str) -> pd.DataFrame:
    """Rename columns to the standard schema (works for Kaggle crypto files)."""
    mapping, used = {}, set()
    for col in df.columns:
        key = _ALIASES.get(re.sub(r"[^a-z]", "", str(col).lower()))
        if key and key not in used:
            mapping[col] = key
            used.add(key)
    df = df.rename(columns=mapping)
    if "symbol" not in df:
        df["symbol"] = df["name"] if "name" in df else default_symbol
    if "market_cap" not in df:
        df["market_cap"] = np.nan
    missing = [c for c in ["date", "open", "high", "low", "close", "volume"] if c not in df]
    if missing:
        raise ValueError(f"Required columns not found: {missing}. Found: {list(df.columns)}")
    return df[BASE_COLS]


def load_data(path: str) -> pd.DataFrame:
    """Load one CSV (all coins) or a folder of per-coin CSV files."""
    if os.path.isdir(path):
        files = sorted(glob.glob(os.path.join(path, "**", "*.csv"), recursive=True))
    else:
        files = [path]
    if not files:
        raise FileNotFoundError(f"No CSV files found at {path}")
    frames = []
    for f in files:
        stem = os.path.splitext(os.path.basename(f))[0].replace("coin_", "")
        frames.append(_standardize(pd.read_csv(f), stem))
    return pd.concat(frames, ignore_index=True)


# --------------------------------------------------------------------------
# 2. DATA CLEANING
# --------------------------------------------------------------------------
def clean_data(df: pd.DataFrame, min_rows: int = 120):
    """Return (clean_df, report). Handles types, duplicates, invalid and missing values."""
    df = df.copy()
    report = {"rows_raw": int(len(df)), "missing_before": {k: int(v) for k, v in df.isna().sum().items()}}

    df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True).dt.tz_localize(None).dt.normalize()
    for c in ["open", "high", "low", "close", "volume", "market_cap"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["symbol"] = df["symbol"].astype(str).str.strip().str.upper()
    df = df.dropna(subset=["date", "symbol"])

    n = len(df)
    df = df.drop_duplicates(["symbol", "date"], keep="last")
    report["duplicates_removed"] = int(n - len(df))

    df = df.sort_values(["symbol", "date"]).reset_index(drop=True)

    # invalid prices: NaN, non-positive, or high < low
    ohlc = df[["open", "high", "low", "close"]]
    bad = ohlc.isna().any(axis=1) | (ohlc <= 0).any(axis=1) | (df["high"] < df["low"])
    report["invalid_price_rows_removed"] = int(bad.sum())
    df = df[~bad].copy()

    # volume / market cap: negatives or zeros are invalid -> NaN -> fill within each coin
    df.loc[df["volume"] < 0, "volume"] = np.nan
    df.loc[df["market_cap"] <= 0, "market_cap"] = np.nan
    for c in ["volume", "market_cap"]:
        df[c] = df.groupby("symbol")[c].transform(lambda s: s.ffill().bfill())
    n = len(df)
    df = df.dropna(subset=["volume", "market_cap"])
    report["rows_dropped_no_volume_or_mcap"] = int(n - len(df))

    # keep coins with enough history for rolling windows
    counts = df["symbol"].value_counts()
    keep = counts[counts >= min_rows].index
    report["coins_dropped_short_history"] = int((counts < min_rows).sum())
    df = df[df["symbol"].isin(keep)].reset_index(drop=True)
    if df.empty:
        raise ValueError(f"No usable data after cleaning: each coin needs at least {min_rows} valid daily rows "
                         "with price, volume and market cap.")

    report.update(rows_clean=int(len(df)), coins=int(df["symbol"].nunique()),
                  date_min=str(df["date"].min().date()), date_max=str(df["date"].max().date()))
    return df, report


# --------------------------------------------------------------------------
# 3. FEATURE ENGINEERING
# --------------------------------------------------------------------------
FEATURES = [
    "log_return", "abs_return", "oc_return", "hl_range", "parkinson_7",
    "vol_7", "vol_14", "vol_30", "vol_ratio_7_30", "vol_7_lag7",
    "ret_mean_7", "ret_mean_14", "ret_mean_30",
    "close_sma7_ratio", "close_sma14_ratio", "close_sma30_ratio", "macd_pct",
    "bb_width", "bb_pctb", "atr14_pct", "rsi14",
    "volume_change", "volume_sma7_ratio", "liquidity_ratio", "liquidity_ratio_7",
    "log_volume", "log_mcap", "day_of_week",
]

FEATURE_DESCRIPTIONS = {
    "log_return": "Daily log return ln(close_t / close_t-1)",
    "abs_return": "Absolute daily log return",
    "oc_return": "Intraday log return ln(close / open)",
    "hl_range": "(High - Low) / Close, daily price range",
    "parkinson_7": "7-day Parkinson volatility estimator (uses High/Low)",
    "vol_7": "Rolling std of log returns, 7 days",
    "vol_14": "Rolling std of log returns, 14 days",
    "vol_30": "Rolling std of log returns, 30 days",
    "vol_ratio_7_30": "Short-term vs long-term volatility (vol_7 / vol_30)",
    "vol_7_lag7": "vol_7 seven days ago (volatility persistence)",
    "ret_mean_7": "Rolling mean log return, 7 days", "ret_mean_14": "Rolling mean log return, 14 days",
    "ret_mean_30": "Rolling mean log return, 30 days",
    "close_sma7_ratio": "Close / 7-day moving average - 1", "close_sma14_ratio": "Close / 14-day moving average - 1",
    "close_sma30_ratio": "Close / 30-day moving average - 1",
    "macd_pct": "(EMA12 - EMA26) / Close, trend momentum",
    "bb_width": "Bollinger Band width (20d, 2 std) / middle band",
    "bb_pctb": "Bollinger %B: position of close inside the bands",
    "atr14_pct": "Average True Range (14d) / Close",
    "rsi14": "Relative Strength Index (14d)",
    "volume_change": "Daily change in log(1 + volume)",
    "volume_sma7_ratio": "Volume / 7-day average volume",
    "liquidity_ratio": "Volume / Market cap",
    "liquidity_ratio_7": "7-day average of Volume / Market cap",
    "log_volume": "log(1 + volume)", "log_mcap": "log(1 + market cap), coin size",
    "day_of_week": "Day of week (0 = Monday)",
}


def _coin_features(g: pd.DataFrame) -> pd.DataFrame:
    g = g.sort_values("date").reset_index(drop=True)
    o, h, l, c, v, m = (g[k] for k in ["open", "high", "low", "close", "volume", "market_cap"])
    r = np.log(c).diff()
    f = pd.DataFrame({"date": g["date"], "symbol": g["symbol"], "close": c})

    f["log_return"] = r
    f["abs_return"] = r.abs()
    f["oc_return"] = np.log(c / o)
    f["hl_range"] = (h - l) / c
    park = np.log(h / l) ** 2 / (4 * np.log(2))
    f["parkinson_7"] = np.sqrt(park.rolling(7).mean())

    for w in (7, 14, 30):
        f[f"vol_{w}"] = r.rolling(w).std()
        f[f"ret_mean_{w}"] = r.rolling(w).mean()
        f[f"close_sma{w}_ratio"] = c / c.rolling(w).mean() - 1
    f["vol_ratio_7_30"] = f["vol_7"] / f["vol_30"]
    f["vol_7_lag7"] = f["vol_7"].shift(7)

    ema12, ema26 = c.ewm(span=12, adjust=False).mean(), c.ewm(span=26, adjust=False).mean()
    f["macd_pct"] = (ema12 - ema26) / c

    mid, sd = c.rolling(20).mean(), c.rolling(20).std()
    f["bb_width"] = 4 * sd / mid
    f["bb_pctb"] = (c - (mid - 2 * sd)) / (4 * sd)

    prev = c.shift(1)
    tr = pd.concat([h - l, (h - prev).abs(), (l - prev).abs()], axis=1).max(axis=1)
    f["atr14_pct"] = tr.rolling(14).mean() / c

    d = c.diff()
    gain, loss = d.clip(lower=0).rolling(14).mean(), (-d.clip(upper=0)).rolling(14).mean()
    rsi = np.where(loss == 0, np.where(gain > 0, 100.0, 50.0), 100 - 100 / (1 + gain / loss))
    f["rsi14"] = pd.Series(rsi, index=f.index).where(gain.notna() & loss.notna())

    f["volume_change"] = np.log1p(v).diff()
    f["volume_sma7_ratio"] = v / v.rolling(7).mean()
    f["liquidity_ratio"] = v / m
    f["liquidity_ratio_7"] = f["liquidity_ratio"].rolling(7).mean()
    f["log_volume"] = np.log1p(v)
    f["log_mcap"] = np.log1p(m)
    f["day_of_week"] = g["date"].dt.dayofweek

    # TARGET: std of the next HORIZON daily log returns (t+1 ... t+HORIZON)
    f["target_vol"] = r.rolling(HORIZON).std().shift(-HORIZON)
    return f


def build_features(clean: pd.DataFrame) -> pd.DataFrame:
    """Compute features (+ target) for every coin. Rows with NaNs are kept; filter before training."""
    out = pd.concat([_coin_features(g) for _, g in clean.groupby("symbol")], ignore_index=True)
    return out.replace([np.inf, -np.inf], np.nan)


def make_model_frame(feat: pd.DataFrame) -> pd.DataFrame:
    return feat.dropna(subset=FEATURES + ["target_vol"]).sort_values(["date", "symbol"]).reset_index(drop=True)


def time_split(model_df: pd.DataFrame, test_frac: float = 0.2, gap: int = HORIZON):
    """Chronological split. A `gap` of days is removed from the end of train so that the
    7-day-ahead targets of the last training rows do not overlap the test period."""
    dates = np.sort(model_df["date"].unique())
    cutoff = pd.Timestamp(dates[int(len(dates) * (1 - test_frac))])
    train = model_df[model_df["date"] < cutoff - pd.Timedelta(days=gap)]
    test = model_df[model_df["date"] >= cutoff]
    return train.reset_index(drop=True), test.reset_index(drop=True)


# --------------------------------------------------------------------------
# 4. MODELS & EVALUATION
# --------------------------------------------------------------------------
def evaluate(y_true, y_pred) -> dict:
    return {"RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
            "MAE": float(mean_absolute_error(y_true, y_pred)),
            "R2": float(r2_score(y_true, y_pred))}


def get_models() -> dict:
    scaler = lambda: ("scaler", StandardScaler())
    return {
        "Ridge Regression": Pipeline([scaler(), ("model", Ridge(alpha=1.0))]),
        "Random Forest": Pipeline([scaler(), ("model", RandomForestRegressor(
            n_estimators=150, max_depth=10, min_samples_leaf=20, max_features=0.5,
            n_jobs=-1, random_state=RANDOM_STATE))]),
        "Hist Gradient Boosting": Pipeline([scaler(), ("model", HistGradientBoostingRegressor(
            random_state=RANDOM_STATE))]),
    }


PARAM_GRIDS = {
    "Ridge Regression": {"model__alpha": [0.01, 0.1, 1, 10, 100]},
    "Random Forest": {"model__n_estimators": [100, 200], "model__max_depth": [6, 10, 14],
                      "model__min_samples_leaf": [10, 20, 50], "model__max_features": [0.3, 0.5, 0.8]},
    "Hist Gradient Boosting": {"model__learning_rate": [0.03, 0.05, 0.1], "model__max_iter": [200, 400],
                               "model__max_leaf_nodes": [15, 31, 63], "model__min_samples_leaf": [20, 50, 100],
                               "model__l2_regularization": [0.0, 0.1, 1.0]},
}


def _risk_level(value: float, q33: float, q66: float) -> str:
    return "Low" if value < q33 else ("Medium" if value < q66 else "High")


# --------------------------------------------------------------------------
# 5. EDA (writes figures + reports/EDA_REPORT.md from the real data)
# --------------------------------------------------------------------------
def _md_table(df: pd.DataFrame, floatfmt: str = "{:.4f}") -> str:
    cols = list(df.columns)
    lines = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    for _, row in df.iterrows():
        cells = [floatfmt.format(x) if isinstance(x, (float, np.floating)) else str(x) for x in row]
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)


def run_eda(clean: pd.DataFrame, feat: pd.DataFrame, clean_report: dict, report_dir: str = "reports"):
    fig_dir = os.path.join(report_dir, "figures")
    os.makedirs(fig_dir, exist_ok=True)
    sns.set_theme(style="whitegrid")
    mdl = make_model_frame(feat)

    # ---- statistics ----
    stats = clean[["open", "high", "low", "close", "volume", "market_cap"]].describe().T
    stats = stats[["mean", "std", "min", "50%", "max"]].reset_index().rename(columns={"index": "column"})
    coin_vol = (feat.groupby("symbol")["log_return"].std().sort_values(ascending=False)
                .rename("daily_return_std").reset_index())
    corr = (mdl[FEATURES + ["target_vol"]].corr()["target_vol"].drop("target_vol")
            .sort_values(key=abs, ascending=False))
    mkt = mdl.groupby("date")["vol_7"].mean()

    # ---- figures ----
    top = clean.groupby("symbol")["market_cap"].mean().nlargest(5).index
    plt.figure(figsize=(10, 5))
    for s in top:
        d = clean[clean.symbol == s]
        plt.plot(d["date"], d["close"] / d["close"].iloc[0], label=s)
    plt.yscale("log"); plt.title("Price trend of top 5 coins by market cap (indexed to 1, log scale)")
    plt.legend(); plt.tight_layout(); plt.savefig(f"{fig_dir}/price_trends.png", dpi=110); plt.close()

    plt.figure(figsize=(8, 4))
    sns.histplot(feat["log_return"].dropna().clip(-0.4, 0.4), bins=100, kde=True)
    plt.title("Distribution of daily log returns (clipped at +/-0.4)"); plt.tight_layout()
    plt.savefig(f"{fig_dir}/return_distribution.png", dpi=110); plt.close()

    plt.figure(figsize=(8, 4))
    sns.histplot(mdl["target_vol"], bins=80, kde=True)
    plt.title(f"Distribution of target: next-{HORIZON}-day volatility"); plt.tight_layout()
    plt.savefig(f"{fig_dir}/target_distribution.png", dpi=110); plt.close()

    plt.figure(figsize=(9, 5))
    sub = coin_vol.head(15)
    sns.barplot(data=sub, y="symbol", x="daily_return_std", color="tab:red")
    plt.title("15 most volatile coins (std of daily log returns)"); plt.tight_layout()
    plt.savefig(f"{fig_dir}/volatility_by_coin.png", dpi=110); plt.close()

    plt.figure(figsize=(10, 4))
    plt.plot(mkt.index, mkt.values)
    plt.title("Market-wide average 7-day volatility over time"); plt.ylabel("avg vol_7"); plt.tight_layout()
    plt.savefig(f"{fig_dir}/market_volatility_over_time.png", dpi=110); plt.close()

    key = corr.head(12).index.tolist() + ["target_vol"]
    plt.figure(figsize=(9, 7))
    sns.heatmap(mdl[key].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, annot_kws={"size": 7})
    plt.title("Correlation: top features vs target"); plt.tight_layout()
    plt.savefig(f"{fig_dir}/correlation_heatmap.png", dpi=110); plt.close()

    dow = mdl.groupby("day_of_week")["target_vol"].mean()
    plt.figure(figsize=(7, 4)); dow.plot(kind="bar")
    plt.title("Average next-7-day volatility by day of week (0 = Mon)"); plt.tight_layout()
    plt.savefig(f"{fig_dir}/volatility_by_weekday.png", dpi=110); plt.close()

    # ---- markdown report ----
    peak = mkt.nlargest(3)
    md = f"""# Exploratory Data Analysis Report
*Generated automatically by `run_eda` from the dataset that was actually used.*

## 1. Dataset summary
- Rows after cleaning: **{clean_report['rows_clean']:,}** (raw: {clean_report['rows_raw']:,})
- Cryptocurrencies: **{clean_report['coins']}**
- Date range: **{clean_report['date_min']}** to **{clean_report['date_max']}**
- Duplicates removed: {clean_report['duplicates_removed']:,}; invalid price rows removed: {clean_report['invalid_price_rows_removed']:,}
- Rows dropped for missing volume/market cap: {clean_report['rows_dropped_no_volume_or_mcap']:,}; coins dropped (short history): {clean_report['coins_dropped_short_history']}

### Missing values before cleaning
{_md_table(pd.DataFrame({'column': list(clean_report['missing_before']), 'missing': list(clean_report['missing_before'].values())}), '{:.0f}')}

## 2. Descriptive statistics
{_md_table(stats, '{:,.4f}')}

## 3. Volatility insights
- Most volatile coins (std of daily log return): {', '.join(f"{r.symbol} ({r.daily_return_std:.3f})" for r in coin_vol.head(5).itertuples())}
- Least volatile coins: {', '.join(f"{r.symbol} ({r.daily_return_std:.3f})" for r in coin_vol.tail(5).itertuples())}
- Market-wide volatility peaked around: {', '.join(str(d.date()) for d in peak.index)}
- Mean next-{HORIZON}-day volatility: {mdl['target_vol'].mean():.4f}; median {mdl['target_vol'].median():.4f}; 95th percentile {mdl['target_vol'].quantile(0.95):.4f} (right-skewed: calm periods with occasional spikes)
- Daily log returns: skewness {feat['log_return'].skew():.2f}, excess kurtosis {feat['log_return'].kurt():.1f} (fat tails)

## 4. Features most correlated with the target (Pearson)
{_md_table(corr.head(10).rename('correlation').reset_index().rename(columns={'index': 'feature'}), '{:.3f}')}

Volatility clusters in time, so lagged volatility and price-range measures (`vol_7`, `parkinson_7`, `atr14_pct`, ...) are usually strong predictors.

## 5. Figures
![Price trends](figures/price_trends.png)
![Return distribution](figures/return_distribution.png)
![Target distribution](figures/target_distribution.png)
![Volatility by coin](figures/volatility_by_coin.png)
![Market volatility over time](figures/market_volatility_over_time.png)
![Correlation heatmap](figures/correlation_heatmap.png)
![Volatility by weekday](figures/volatility_by_weekday.png)
"""
    with open(os.path.join(report_dir, "EDA_REPORT.md"), "w", encoding="utf-8") as fh:
        fh.write(md)
    return {"stats": stats, "coin_vol": coin_vol, "corr": corr, "market_vol": mkt}


# --------------------------------------------------------------------------
# 6. TRAINING
# --------------------------------------------------------------------------
def train_pipeline(clean: pd.DataFrame, clean_report: dict, out_dir: str = "artifacts",
                   report_dir: str = "reports", tune: bool = True, n_iter: int = 8,
                   test_frac: float = 0.2, verbose: bool = True):
    os.makedirs(out_dir, exist_ok=True)
    fig_dir = os.path.join(report_dir, "figures")
    os.makedirs(fig_dir, exist_ok=True)
    log = print if verbose else (lambda *a, **k: None)

    feat = build_features(clean)
    mdl = make_model_frame(feat)
    train, test = time_split(mdl, test_frac)
    X_tr, y_tr, X_te, y_te = train[FEATURES], train["target_vol"], test[FEATURES], test["target_vol"]
    log(f"Rows -> train: {len(train):,} | test: {len(test):,} | coins: {mdl['symbol'].nunique()}")
    log(f"Train: {train['date'].min().date()} to {train['date'].max().date()} | "
        f"Test: {test['date'].min().date()} to {test['date'].max().date()}")

    # time-series CV on training data only (gap avoids overlapping 7-day targets)
    n_coins = mdl["symbol"].nunique()
    cv = TimeSeriesSplit(n_splits=3, gap=HORIZON * n_coins)

    rows = [{"Model": "Baseline: persistence (vol_7)", "CV_RMSE": np.nan, **evaluate(y_te, test["vol_7"])}]
    fitted, cv_scores = {}, {}
    for name, pipe in get_models().items():
        cv_rmse = -cross_val_score(pipe, X_tr, y_tr, cv=cv, scoring="neg_root_mean_squared_error").mean()
        pipe.fit(X_tr, y_tr)
        fitted[name], cv_scores[name] = pipe, cv_rmse
        rows.append({"Model": name, "CV_RMSE": cv_rmse, **evaluate(y_te, pipe.predict(X_te))})
        log(f"{name:<24} CV RMSE={cv_rmse:.5f} | test RMSE={rows[-1]['RMSE']:.5f} R2={rows[-1]['R2']:.3f}")

    best_name = min(cv_scores, key=cv_scores.get)          # selected on CV only (no test peeking)
    best = fitted[best_name]
    tuning = {"selected_model": best_name, "tuned": False}
    if tune:
        log(f"Tuning {best_name} with RandomizedSearchCV (n_iter={n_iter}) ...")
        search = RandomizedSearchCV(get_models()[best_name], PARAM_GRIDS[best_name], n_iter=n_iter, cv=cv,
                                    scoring="neg_root_mean_squared_error", random_state=RANDOM_STATE,
                                    n_jobs=1, refit=True)
        search.fit(X_tr, y_tr)
        best = search.best_estimator_
        tuning.update(tuned=True, best_params={k: (v.item() if hasattr(v, "item") else v)
                                               for k, v in search.best_params_.items()},
                      tuned_cv_rmse=float(-search.best_score_), untuned_cv_rmse=float(cv_scores[best_name]))
        rows.append({"Model": f"{best_name} (tuned)", "CV_RMSE": -search.best_score_,
                     **evaluate(y_te, best.predict(X_te))})
        log("Best params:", tuning["best_params"])

    comparison = pd.DataFrame(rows)
    final_pred = best.predict(X_te)
    final_metrics = evaluate(y_te, final_pred)
    log("FINAL (held-out test):", {k: round(v, 5) for k, v in final_metrics.items()})

    # ---- importance + plots ----
    samp = test.sample(min(3000, len(test)), random_state=RANDOM_STATE)
    imp = permutation_importance(best, samp[FEATURES], samp["target_vol"], n_repeats=3,
                                 random_state=RANDOM_STATE, scoring="neg_root_mean_squared_error")
    importance = (pd.Series(imp.importances_mean, index=FEATURES).sort_values(ascending=False)
                  .rename("importance").reset_index().rename(columns={"index": "feature"}))

    sns.set_theme(style="whitegrid")
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    lim = float(np.quantile(np.r_[y_te, final_pred], 0.995))
    ax[0].scatter(y_te, final_pred, s=4, alpha=0.3); ax[0].plot([0, lim], [0, lim], "r--")
    ax[0].set(xlim=(0, lim), ylim=(0, lim), xlabel="Actual volatility", ylabel="Predicted volatility",
              title="Actual vs Predicted (test)")
    sns.histplot(y_te - final_pred, bins=80, ax=ax[1]); ax[1].set_title("Residuals (actual - predicted)")
    plt.tight_layout(); plt.savefig(f"{fig_dir}/actual_vs_predicted.png", dpi=110); plt.close()

    tdf = test.assign(pred=final_pred).groupby("date")[["target_vol", "pred"]].mean()
    plt.figure(figsize=(11, 4)); plt.plot(tdf.index, tdf["target_vol"], label="Actual (avg of coins)")
    plt.plot(tdf.index, tdf["pred"], label="Predicted", alpha=0.85)
    plt.title("Market-average volatility: actual vs predicted on test period"); plt.legend(); plt.tight_layout()
    plt.savefig(f"{fig_dir}/test_timeseries.png", dpi=110); plt.close()

    plt.figure(figsize=(8, 7)); sns.barplot(data=importance.head(15), y="feature", x="importance", color="tab:blue")
    plt.title("Permutation importance (increase in RMSE when shuffled)"); plt.tight_layout()
    plt.savefig(f"{fig_dir}/feature_importance.png", dpi=110); plt.close()

    # ---- save artefacts ----
    q33, q66 = float(y_tr.quantile(1 / 3)), float(y_tr.quantile(2 / 3))
    bundle = {"model": best, "features": FEATURES, "horizon": HORIZON, "model_name": best_name,
              "risk_thresholds": {"q33": q33, "q66": q66}, "test_metrics": final_metrics,
              "train_range": [str(train["date"].min().date()), str(train["date"].max().date())]}
    joblib.dump(bundle, os.path.join(out_dir, "volatility_model.joblib"))
    comparison.to_csv(os.path.join(report_dir, "model_comparison.csv"), index=False)
    importance.to_csv(os.path.join(report_dir, "feature_importance.csv"), index=False)
    with open(os.path.join(report_dir, "results.json"), "w") as fh:
        json.dump({"final_metrics": final_metrics, "tuning": tuning, "clean_report": clean_report,
                   "risk_thresholds": bundle["risk_thresholds"]}, fh, indent=2, default=str)

    write_final_report(report_dir, clean_report, comparison, importance, final_metrics, tuning,
                       bundle, len(train), len(test), (train, test))
    log(f"Saved model to {out_dir}/volatility_model.joblib and reports to {report_dir}/")
    return bundle, comparison, importance


def write_final_report(report_dir, clean_report, comparison, importance, metrics, tuning, bundle,
                       n_train, n_test, splits):
    train, test = splits
    base = comparison.iloc[0]
    improve = (1 - metrics["RMSE"] / base["RMSE"]) * 100
    top5 = importance["feature"].head(5).tolist()
    range_like = [f for f in top5 if f.startswith(("vol_", "parkinson", "hl_range", "atr", "bb_width", "abs_return"))]
    top_feats = ", ".join(f"`{f}`" for f in importance["feature"].head(5))
    tune_txt = (f"Hyper-parameters were tuned with `RandomizedSearchCV` + `TimeSeriesSplit`; best parameters: "
                f"`{tuning['best_params']}`." if tuning["tuned"] else "No hyper-parameter tuning was run.")
    md = f"""# Final Report: Cryptocurrency Volatility Prediction
*Generated automatically after training, all numbers below come from the run that produced `artifacts/volatility_model.joblib`.*

## 1. Objective
Predict how volatile each cryptocurrency will be over the **next {HORIZON} days** so traders and risk teams can
anticipate turbulent periods. Volatility = standard deviation of daily log returns.

## 2. Data
- {clean_report['coins']} coins, {clean_report['rows_clean']:,} daily rows, {clean_report['date_min']} to {clean_report['date_max']}.
- Cleaning: parsed dates and numerics, removed {clean_report['duplicates_removed']:,} duplicates and
  {clean_report['invalid_price_rows_removed']:,} invalid price rows, forward/back-filled volume and market cap within each coin,
  dropped {clean_report['coins_dropped_short_history']} coins with short history.
- Full statistics and plots: `reports/EDA_REPORT.md`.

## 3. Methodology
1. **Feature engineering** ({len(FEATURES)} features): returns, rolling volatility (7/14/30d), Parkinson estimator, ATR, Bollinger
   Bands, RSI, MACD, moving-average ratios, volume change and liquidity ratio (volume / market cap), coin size, weekday.
   See `docs/LLD.md` for each feature.
2. **Target**: std of log returns over days t+1 ... t+{HORIZON}; features only use data up to day t (no leakage).
3. **Split**: chronological. Train {train['date'].min().date()} to {train['date'].max().date()} ({n_train:,} rows);
   test {test['date'].min().date()} to {test['date'].max().date()} ({n_test:,} rows). A {HORIZON}-day gap prevents overlap.
4. **Scaling**: `StandardScaler` inside each sklearn `Pipeline`, fitted on training data only.
5. **Models compared**: persistence baseline, Ridge, Random Forest, Hist Gradient Boosting; selected with time-series CV
   on training data. Selected: **{tuning['selected_model']}**. {tune_txt}

## 4. Results (held-out test set)
{_md_table(comparison, '{:.5f}')}

**Final model ({tuning['selected_model']}{' tuned' if tuning['tuned'] else ''}):**
RMSE = **{metrics['RMSE']:.5f}**, MAE = **{metrics['MAE']:.5f}**, R² = **{metrics['R2']:.3f}**.
Test RMSE is {abs(improve):.1f}% {'lower' if improve >= 0 else 'higher'} than the persistence baseline (which simply reuses the last 7-day volatility).

![Actual vs predicted](figures/actual_vs_predicted.png)
![Test period](figures/test_timeseries.png)
![Feature importance](figures/feature_importance.png)

## 5. Key insights
- Most important features: {top_feats}.
- Risk bands used by the app (from training distribution of the target): Low < {bundle['risk_thresholds']['q33']:.4f} <= Medium < {bundle['risk_thresholds']['q66']:.4f} <= High.
- {len(range_like)} of the top 5 features are recent price-range / volatility measures ({', '.join(f'`{f}`' for f in range_like) or 'none'}), consistent with volatility clustering in time.

## 6. Limitations
- Crypto markets are driven by news and regulation events that price/volume data cannot foresee; extreme spikes are under-predicted.
- Daily data only; intraday data and sentiment/on-chain features could improve accuracy.
- Model is trained on historical regimes; monitor error and retrain regularly.

## 7. Deployment
`app.py` (Streamlit) and `api.py` (Flask) load the saved bundle and predict next-{HORIZON}-day volatility from the
latest OHLCV + market cap history of any coin. See `README.md`.
"""
    with open(os.path.join(report_dir, "FINAL_REPORT.md"), "w", encoding="utf-8") as fh:
        fh.write(md)


# --------------------------------------------------------------------------
# 7. INFERENCE
# --------------------------------------------------------------------------
def load_bundle(path: str = "artifacts/volatility_model.joblib") -> dict:
    return joblib.load(path)


def predict_latest(history: pd.DataFrame, bundle: dict) -> dict:
    """Predict the next-HORIZON-day volatility for ONE coin from its daily history.

    `history` needs columns: date, open, high, low, close, volume, market_cap
    (at least ~60 daily rows; the last row is the most recent day)."""
    df = _standardize(history.copy(), "COIN")      # adds symbol="COIN" if the column is missing
    clean, _ = clean_data(df, min_rows=40)
    feat = build_features(clean)
    last = feat.dropna(subset=bundle["features"]).tail(1)
    if last.empty:
        raise ValueError("Not enough clean history to compute all features (need roughly 60+ daily rows).")
    pred = float(bundle["model"].predict(last[bundle["features"]])[0])
    q = bundle["risk_thresholds"]
    return {"as_of_date": str(last["date"].iloc[0].date()), "horizon_days": bundle["horizon"],
            "predicted_daily_volatility": pred, "annualized_volatility": pred * np.sqrt(365),
            "risk_level": _risk_level(pred, q["q33"], q["q66"]),
            "current_vol_7": float(last["vol_7"].iloc[0])}


In [ ]:
import sys
sys.path.insert(0, 'src')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
import crypto_volatility as cv

DATA_PATH = 'data'      # <-- folder with the dataset CSV(s), or a single CSV file path
print('Prediction horizon:', cv.HORIZON, 'days')

## **Step 1: Data Collection**
Load the historical OHLC, volume and market-cap data. Works with one combined CSV or one CSV per coin; column names are matched automatically (e.g. `Marketcap` → `market_cap`).

In [ ]:
raw = cv.load_data(DATA_PATH)
print('Raw shape:', raw.shape)
display(raw.head())
print(raw.dtypes)

## **Step 2: Data Preprocessing**
Handles data types, duplicates, invalid prices (≤ 0 or High < Low), missing volume/market cap (filled within each coin), and drops coins with too little history.
Scaling is done later inside the model Pipeline (`StandardScaler`, fitted on training data only) to avoid leakage.

In [ ]:
clean, report = cv.clean_data(raw)
for k, v in report.items():
    print(f'{k}: {v}')
display(clean.describe().T)

## **Step 3: Exploratory Data Analysis (EDA)**
Statistics, trends, correlations and distributions. Figures are saved in `reports/figures/` and a written report in `reports/EDA_REPORT.md`.

In [ ]:
feat = cv.build_features(clean)
eda = cv.run_eda(clean, feat, report)

print('Most volatile coins (std of daily log return):')
display(eda['coin_vol'].head(10))
print('Top features correlated with next-7-day volatility:')
display(eda['corr'].head(10))

In [ ]:
for f in ['price_trends', 'return_distribution', 'target_distribution', 'volatility_by_coin',
          'market_volatility_over_time', 'correlation_heatmap', 'volatility_by_weekday']:
    display(Image(f'reports/figures/{f}.png'))

**EDA observations** are written automatically from your data in `reports/EDA_REPORT.md`:

In [ ]:
display(Markdown(open('reports/EDA_REPORT.md', encoding='utf-8').read().split('## 5. Figures')[0]))

## **Step 4: Feature Engineering**
New features capture **volatility**, **trend/momentum** and **liquidity**. Every feature at day *t* uses data up to day *t* only.

**Target** = standard deviation of daily log returns over the next 7 days (t+1 … t+7).

| Feature | Meaning |
|---|---|
| `log_return` | Daily log return ln(close_t / close_t-1) |
| `abs_return` | Absolute daily log return |
| `oc_return` | Intraday log return ln(close / open) |
| `hl_range` | (High - Low) / Close, daily price range |
| `parkinson_7` | 7-day Parkinson volatility estimator (uses High/Low) |
| `vol_7` | Rolling std of log returns, 7 days |
| `vol_14` | Rolling std of log returns, 14 days |
| `vol_30` | Rolling std of log returns, 30 days |
| `vol_ratio_7_30` | Short-term vs long-term volatility (vol_7 / vol_30) |
| `vol_7_lag7` | vol_7 seven days ago (volatility persistence) |
| `ret_mean_7` | Rolling mean log return, 7 days |
| `ret_mean_14` | Rolling mean log return, 14 days |
| `ret_mean_30` | Rolling mean log return, 30 days |
| `close_sma7_ratio` | Close / 7-day moving average - 1 |
| `close_sma14_ratio` | Close / 14-day moving average - 1 |
| `close_sma30_ratio` | Close / 30-day moving average - 1 |
| `macd_pct` | (EMA12 - EMA26) / Close, trend momentum |
| `bb_width` | Bollinger Band width (20d, 2 std) / middle band |
| `bb_pctb` | Bollinger %B: position of close inside the bands |
| `atr14_pct` | Average True Range (14d) / Close |
| `rsi14` | Relative Strength Index (14d) |
| `volume_change` | Daily change in log(1 + volume) |
| `volume_sma7_ratio` | Volume / 7-day average volume |
| `liquidity_ratio` | Volume / Market cap |
| `liquidity_ratio_7` | 7-day average of Volume / Market cap |
| `log_volume` | log(1 + volume) |
| `log_mcap` | log(1 + market cap), coin size |
| `day_of_week` | Day of week (0 = Monday) |

In [ ]:
one = feat[feat.symbol == feat.symbol.iloc[0]]
display(one[['date', 'symbol', 'close'] + cv.FEATURES[:8] + ['target_vol']].dropna().tail())
model_df = cv.make_model_frame(feat)
print('Rows usable for modelling:', len(model_df), '| features:', len(cv.FEATURES))

## **Step 5: Model Selection, Training, Tuning and Evaluation**
- **Split**: chronological (earlier 80% train, latest 20% test) with a 7-day gap, because random splits would leak the future.
- **Models**: persistence baseline (reuse last 7-day volatility), Ridge Regression, Random Forest, Hist Gradient Boosting.
- **Scaling**: `StandardScaler` inside each Pipeline.
- **Selection**: time-series cross-validation on the training data only; the best model is tuned with `RandomizedSearchCV`.
- **Metrics**: RMSE, MAE, R² on the unseen test set.

*(This cell can take a few minutes on the full dataset. Set `tune=False` for a quick run.)*

In [ ]:
bundle, comparison, importance = cv.train_pipeline(clean, report, tune=True, n_iter=8)

In [ ]:
print('Model comparison on the held-out test set:')
display(comparison.round(5))
print('Final selected model:', bundle['model_name'])
print({k: round(v, 5) for k, v in bundle['test_metrics'].items()})

## **Step 6: Results and Interpretation**

In [ ]:
for f in ['actual_vs_predicted', 'test_timeseries', 'feature_importance']:
    display(Image(f'reports/figures/{f}.png'))
display(importance.head(10))

## **Step 7: Testing on Unseen Data / Prediction**
Use the saved model to forecast the next 7 days for the latest data of any coin (this is what the Streamlit app and Flask API do).

In [ ]:
bundle = cv.load_bundle('artifacts/volatility_model.joblib')
coin = clean['symbol'].value_counts().index[0]
history = clean[clean.symbol == coin].tail(120)
result = cv.predict_latest(history, bundle)
print('Coin:', coin)
for k, v in result.items():
    print(f'{k}: {v}')

## **Step 8: Deployment (local)**
Run these in a terminal from the repository root (not inside the notebook):
```bash
streamlit run app.py        # web interface: upload a coin CSV and get the 7-day volatility forecast
python api.py               # Flask REST API:  curl -F "file=@coin.csv" http://127.0.0.1:5000/predict
```

## **Step 9: Final Report**
The final report (methodology, results table, insights, limitations) is generated automatically with the real numbers from this run:

In [ ]:
display(Markdown(open('reports/FINAL_REPORT.md', encoding='utf-8').read()))